In [1]:
name = ["t1","t2","t3","t4","t5","t6"]

In [2]:
item = [{"Bread","Butter","Jam"},
        {"Bread","Butter"},
        {"Bread","Jam"},
        {"Jam"},
        {"Butter","Jam"},
        {"Bread","Butter","Jam"}
]

In [3]:
import pandas as pd

In [4]:
# Create a dictionary where keys are column names and values are lists
data = {'Name': name, 'Item_Bought': item}

In [5]:
df = pd.DataFrame(data)

In [6]:
df

,Name,Item_Bought
0,t1,"{Bread, Jam, Butter}"
1,t2,"{Bread, Butter}"
2,t3,"{Bread, Jam}"
3,t4,{Jam}
4,t5,"{Jam, Butter}"
5,t6,"{Bread, Jam, Butter}"


Find the Market Basket Analysis regarding the info below

- {"Jam}
- {"Bread"} -> {"Jam"}

In [7]:
# 2. Support Calculation Function
def calculate_support(df, target_items):
    # Ensure target_items is a set for subset comparison
    target_set = set(target_items)
    
    # Count transactions where target_set is a subset of bought items
    occurrence_count = df['Item_Bought'].apply(lambda x: target_set.issubset(x)).sum()
    
    # Calculate support as fraction of total transactions
    return occurrence_count / len(df)

In [8]:
# 3. Examples
sup_jam = calculate_support(df, ['Jam'])
sup_bread_jam = calculate_support(df, ['Bread', 'Jam'])

print(f"Support(Jam): {sup_jam:.2%}")
print(f"Support(Bread, Jam): {sup_bread_jam:.2%}")

Support(Jam): 83.33%
Support(Bread, Jam): 50.00%


Support Meaning,  
- "How frequently does this specific itemset appear in all transactions?"
- For {Jam}, it is 83.33% appear in all of the store transactions.
- For {Bread,Jam}, it is 50.00% appear in all of the store transactions.

In [9]:
def calculate_confidence(df, antecedent, consequent):
    # Set of both items (A and B)
    both_items = set(antecedent) | set(consequent)
    # Set of only the first item (A)
    antecedent_set = set(antecedent)
    
    # Count occurrences
    count_both = df['Item_Bought'].apply(lambda x: both_items.issubset(x)).sum()
    count_antecedent = df['Item_Bought'].apply(lambda x: antecedent_set.issubset(x)).sum()
    
    if count_antecedent == 0:
        return 0
    return count_both / count_antecedent

In [10]:
# Example: Confidence of Bread -> Jam
conf = calculate_confidence(df, ['Bread'], ['Jam'])
print(f"Confidence(Bread -> Jam): {conf:.2%}") # Output: 75.00%

Confidence(Bread -> Jam): 75.00%


- Confidence measures the likelihood that a customer will buy a second item (the "consequent") 
- given that they have already bought a first item (the "antecedent"). 
- It answers the question: 
    - "Of the people who bought Item A, what percentage also bought Item B?".

- Bread -> Jam: 75% confidence (3 out of 4 bread buyers also bought jam).

In [11]:
def calculate_lift(df, antecedent, consequent):
    support_both = calculate_support(df, set(antecedent) | set(consequent))
    support_a = calculate_support(df, antecedent)
    support_b = calculate_support(df, consequent)
    
    if support_a == 0 or support_b == 0:
        return 0
    return support_both / (support_a * support_b)



In [12]:
# Example: Lift of Bread -> Jam
lift_val = calculate_lift(df, ['Bread'], ['Jam'])
print(f"Lift(Bread -> Jam): {lift_val:.2f}")

Lift(Bread -> Jam): 0.90


- Whenever people buy 1 Bread, it is about 0.9 the people will buy Jam.

Rule -> {Bread} -> {Butter}

In [37]:
target_item = ["Bread","Butter"]

In [38]:
sup_bread_butter = calculate_support(df, target_item)

conf_bread_butter = calculate_confidence(df, [target_item[0]], [target_item[1]])

lift_bread_butter = calculate_lift(df, [target_item[0]], [target_item[1]])

In [39]:
def print_result(target_item,sup,conf,lift):

    print(f"Support ({target_item[0]})--> ({target_item[1]}) : {sup:.2%}")

    print(f"For {target_item}, it is {sup:.2%} appear in all of the store transactions.\n")

    print(f"Confidence ({target_item[0]}) --> ({target_item[1]}) : {conf:.2%}")

    print(f"{conf:.2%} of those who bought {target_item[0]} also bought {target_item[1]}. \n")

    print(f"Lift ({target_item[0]}) --> ({target_item[1]}) : {lift:.2f}")

    if lift > 1:
        print(f"Items are positively associated. Buying {target_item[0]} increases the chance of buying {target_item[1]}.")
    
    elif lift == 1:
        print("Items are independent. There is no special relationship; they just happen to be in the same basket by chance.")

    elif lift < 1:
        print(f"Items are negatively associated (substitutes). Buying {target_item[0]} actually decreases the chance of buying {target_item[1]}")


In [40]:
print_result(target_item,sup_bread_butter,conf_bread_butter,lift_bread_butter)

Support (Bread)--> (Butter) : 50.00%
For ['Bread', 'Butter'], it is 50.00% appear in all of the store transactions.

Confidence (Bread) --> (Butter) : 75.00%
75.00% of those who bought Bread also bought Butter. 

Lift (Bread) --> (Butter) : 1.12
Items are positively associated. Buying Bread increases the chance of buying Butter.
